# CPU implementation

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$

This notebook explains how the CPU timing estimates for the classical MCMC algorithm are obtained.

**Table of contents**

1. [Source](#source)
2. [Algorithm](#algorithm)
3. [Timing calculation](#timing-calculation)
4. [Reproducibility](#reproducibility)

<a id="source"></a>
## Source

The code and data are available in the `estimation_timing/cpu` folder.

The folder contains:

* `timing_estimation_cpu.cpp`: C++ source code for the CPU benchmark. It implements both the local single-spin-flip Metropolis kernel and the uniform dense-proposal kernel.
* `timing_estimation_cpu.slurm`: SLURM script used to compile and run the CPU benchmark for the selected values of $n$.
* `log_timing_estimation_cpu_native_20_100k.out`: output log for the native compilation, using `-march=native -mtune=native`, with `N_MODELS = 10` and `N_STEPS = 100000`.
* `log_timing_estimation_cpu_sapphire_20_100k.out`: output log for the Sapphire-Rapids-targeted compilation, using `-march=sapphirerapids -mprefer-vector-width=512`, with `N_MODELS = 10` and `N_STEPS = 100000`.

<a id="algorithm"></a>
## Algorithm

The C++ program benchmarks one attempted Metropolis proposal for each of two proposal rules. Given a current configuration $x$ and a proposed configuration $y$, the proposal is accepted with probability

$$
A_{yx}^{(\beta)}
=
\min\left\{
1,
\exp\left[-\beta\left(H(y)-H(x)\right)\right]
\right\}.
$$

**Note:** The timing estimate does not include pseudorandom-number generation. All random data are generated before the timed region, so the measured runtime isolates the arithmetic kernels.

The implementation uses single-precision `float` values for the fields, couplings, energies, and Metropolis probabilities. Floating-point arithmetic is the natural optimized representation on the CPU and avoids introducing the fixed-point approximation required by the FPGA implementation.

The dense SK instance is stored in the `IsingModel` structure. The local fields are stored in an array of length $n$, while the couplings are stored as a full row-major $n\times n$ matrix. This uses more memory than storing only the upper triangular part, but makes dense row access in the local kernel less expensive.

The spin configuration is stored as an unpacked array with entries in $\{-1,+1\}$. This also uses more memory than a bitstring, but avoids bit extraction inside the dense arithmetic loops.

The `RandPool` helper class precomputes random integers and uniformly distributed random floating-point values. The same random pools are reset before each timed kernel, so random-number generation is excluded from the benchmark.

### CPU local implementation

The local CPU kernel proposes a single spin flip and computes the corresponding energy difference as

$$
\Delta_i H
=
2x_i
\left(
\widetilde h_i
+
\sum_j \widetilde J_{ij}x_j
\right).
$$

This requires $O(n)$ arithmetic operations per attempted proposal because only the dense local field of the selected spin must be evaluated.

### CPU uniform implementation

The uniform CPU kernel samples a complete proposed spin configuration and evaluates its dense SK energy,

$$
H(y)
=
-\sum_i \widetilde h_i y_i
-\sum_{i<j}\widetilde J_{ij}y_i y_j.
$$

Each attempted proposal therefore requires $O(n^2)$ arithmetic operations.

<a id="timing-calculation"></a>
## Timing calculation

The benchmarks were run for $n\in\{64,128,256,512\}$ on the CINECA Leonardo DCGP partition, using a single core of an Intel Xeon Platinum 8358 processor at $2.60$ GHz. Two compiler configurations were tested: a native build and a Sapphire-Rapids-targeted build. The provided SLURM script contains the native configuration; the Sapphire-Rapids-targeted log was generated by replacing the corresponding `CXXFLAGS` line.

For the local proposal, the measured single-step latency is fitted as

$$
\tau_{\mathrm{cpu}}^{\mathrm{loc}}(n)
=
a_{\mathrm{cpu}}^{\mathrm{loc}}
+
b_{\mathrm{cpu}}^{\mathrm{loc}}n.
$$

For the uniform proposal, it is fitted as

$$
\tau_{\mathrm{cpu}}^{\mathrm{unif}}(n)
=
a_{\mathrm{cpu}}^{\mathrm{unif}}
+
b_{\mathrm{cpu}}^{\mathrm{unif}}n
+
c_{\mathrm{cpu}}^{\mathrm{unif}}n^2.
$$

The following code fits the measured total runtimes using non-negative least squares. The single-step coefficients are obtained by dividing the total-runtime coefficients by `N_MODELS * N_STEPS`.


In [1]:
import numpy as np
from scipy.optimize import nnls

N_MODELS = 10
N_STEPS = 100000
N_TRANSITIONS = N_MODELS * N_STEPS

ns = np.array([64, 128, 256, 512], dtype=float)

# Measurements from log_timing_estimation_cpu_native_20_100k.out.
local_time_native = np.array([0.0189299, 0.0254903, 0.0485925, 0.0957941])
uniform_time_native = np.array([0.77873, 1.97, 6.40592, 25.629])

# Measurements from log_timing_estimation_cpu_sapphire_20_100k.out.
local_time_sapphire = np.array([0.0183836, 0.0221172, 0.0399847, 0.0805035])
uniform_time_sapphire = np.array([0.785426, 2.81768, 7.54626, 24.2614])


def fit_nonnegative_polynomial(y: np.ndarray, degree: int) -> np.ndarray:
    """Return non-negative least-squares polynomial coefficients."""
    design_matrix = np.vstack([ns**power for power in range(degree + 1)]).T
    return nnls(design_matrix, y)[0]


measurements = [
    ("native", local_time_native, uniform_time_native),
    ("Sapphire-Rapids-targeted", local_time_sapphire, uniform_time_sapphire),
]

for name, local_time, uniform_time in measurements:
    a_local, b_local = fit_nonnegative_polynomial(local_time, degree=1)
    a_uniform, b_uniform, c_uniform = fit_nonnegative_polynomial(
        uniform_time,
        degree=2,
    )

    print("Compilation configuration:", name)
    print(f"  local total:    {a_local:.3e} + {b_local:.3e} n")
    print(
        "  uniform total:  "
        f"{a_uniform:.3e} + {b_uniform:.3e} n + {c_uniform:.3e} n^2"
    )
    print(
        "  local/step:     "
        f"{a_local / N_TRANSITIONS:.3e} "
        f"+ {b_local / N_TRANSITIONS:.3e} n"
    )
    print(
        "  uniform/step:   "
        f"{a_uniform / N_TRANSITIONS:.3e} "
        f"+ {b_uniform / N_TRANSITIONS:.3e} n "
        f"+ {c_uniform / N_TRANSITIONS:.3e} n^2\n"
    )


Compilation configuration: native
  local total:    5.122e-03 + 1.753e-04 n
  uniform total:  3.024e-01 + 0.000e+00 n + 9.643e-05 n^2
  local/step:     5.122e-09 + 1.753e-10 n
  uniform/step:   3.024e-07 + 0.000e+00 n + 9.643e-11 n^2

Compilation configuration: Sapphire-Rapids-targeted
  local total:    5.959e-03 + 1.429e-04 n
  uniform total:  0.000e+00 + 1.173e-02 n + 6.964e-05 n^2
  local/step:     5.959e-09 + 1.429e-10 n
  uniform/step:   0.000e+00 + 1.173e-08 n + 6.964e-11 n^2



Using the recorded timings, the Sapphire-Rapids-targeted build gives the single-step latency models used in the paper:

$$
\tau_{\mathrm{cpu}}^{\mathrm{loc}}(n)
=
\left(
5.959\times10^{-9}
+
1.429\times10^{-10}n
\right)\,\mathrm{s},
$$

and

$$
\tau_{\mathrm{cpu}}^{\mathrm{unif}}(n)
=
\left(
1.173\times10^{-8}n
+
6.964\times10^{-11}n^2
\right)\,\mathrm{s}.
$$

The fitted constant term for the uniform proposal is zero under the non-negative least-squares constraint. The native build gives comparable scaling, but the resource estimates use the Sapphire-Rapids-targeted coefficients above.

<a id="reproducibility"></a>
## Reproducibility

To reproduce the CPU benchmark, compile `timing_estimation_cpu.cpp` separately for each system size, passing $n$ as a compile-time macro. The tested sizes are $n\in\{64,128,256,512\}$. The recorded logs use `N_MODELS = 10` independent SK instances and `N_STEPS = 100000` attempted Metropolis proposals per instance.

The native compilation uses GCC with the following optimization flags:

```text
-std=c++20 -O3 -march=native -mtune=native -ffast-math -fno-math-errno -funroll-loops -DNDEBUG
```

The Sapphire-Rapids-targeted compilation uses:

```text
-std=c++20 -O3 -march=sapphirerapids -mprefer-vector-width=512 -ffast-math -fno-math-errno -funroll-loops -DNDEBUG
```

The relevant flags have the following roles:

* `-O3` enables aggressive compiler optimizations.
* `-march=native` and `-mtune=native` specialize code generation and instruction scheduling for the host CPU.
* `-march=sapphirerapids` targets the Intel Sapphire Rapids instruction set.
* `-mprefer-vector-width=512` asks GCC to prefer 512-bit vector operations when profitable.
* `-ffast-math` permits non-strict floating-point optimizations.
* `-fno-math-errno` specifies that the program does not inspect `errno` after calls to mathematical functions.
* `-funroll-loops` enables profitable loop unrolling.
* `-DNDEBUG` disables debug assertions.

Run the benchmark using the provided SLURM script:

```bash
sbatch timing_estimation_cpu.slurm
```

The benchmark generates logs analogous to:

```text
log_timing_estimation_cpu_native_20_100k.out
log_timing_estimation_cpu_sapphire_20_100k.out
```

When changing `N_MODELS`, `N_STEPS`, or the tested system sizes in the C++ source or SLURM script, update the fitting code accordingly.
